# Training on saved activations -- when to use this

Use `train_mode=True` when a saved TorchLens activation is an input to a loss that will
feed `backward()`. The common patterns are auxiliary losses, probes on frozen
backbones, multi-tap losses, and teacher-student distillation. For capture-only
workflows, use the fastlog tutorial instead: `notebooks/fastlog_tutorial.ipynb`.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F

repo_root = Path.cwd().resolve()
if not (repo_root / "torchlens").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import torchlens as tl  # noqa: E402


torch.manual_seed(7)


class ResidualBlock(nn.Module):
    """Small residual block used by TinyResNet."""

    def __init__(self, channels: int) -> None:
        """Initialize two convolutions and one reusable ReLU."""

        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Return a residual block output."""

        hidden = self.relu(self.conv1(x))
        return self.relu(self.conv2(hidden) + x)


class TinyResNet(nn.Module):
    """Tiny CPU ResNet-style classifier used for training patterns."""

    def __init__(self, width: int = 4, num_classes: int = 3) -> None:
        """Initialize a model with named tap points under 100k parameters."""

        super().__init__()
        self.stem = nn.Conv2d(1, width, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.block1 = ResidualBlock(width)
        self.block2 = ResidualBlock(width)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.head = nn.Linear(width, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Return logits for a batch of 8x8 grayscale inputs."""

        x = self.relu(self.stem(x))
        x = self.block1(x)
        x = self.block2(x)
        x = self.pool(x).flatten(1)
        return self.head(x)


class TinyTeacher(nn.Module):
    """Frozen teacher network for activation distillation."""

    def __init__(self) -> None:
        """Initialize the teacher backbone."""

        super().__init__()
        self.net = TinyResNet(width=4, num_classes=3)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Return teacher logits."""

        return self.net(x)


class TinyStudent(nn.Module):
    """Trainable student network for activation distillation."""

    def __init__(self) -> None:
        """Initialize the student backbone."""

        super().__init__()
        self.net = TinyResNet(width=4, num_classes=3)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Return student logits."""

        return self.net(x)


def count_parameters(module: nn.Module) -> int:
    """Return the number of parameters in a module."""

    return sum(param.numel() for param in module.parameters())


def require_any_grad(module: nn.Module) -> None:
    """Assert that at least one module parameter has a gradient."""

    assert any(param.grad is not None for param in module.parameters())


def require_no_grad(module: nn.Module) -> None:
    """Assert that no module parameter has a gradient."""

    assert all(param.grad is None for param in module.parameters())


def freeze(module: nn.Module) -> None:
    """Disable gradient updates for all parameters in a module."""

    for param in module.parameters():
        param.requires_grad_(False)


def first_payload(recording: tl.fastlog.Recording, layer_type: str) -> torch.Tensor:
    """Return the first RAM payload in a recording for one layer type."""

    return next(
        record.ram_payload
        for record in recording
        if record.ctx.layer_type == layer_type and record.ram_payload is not None
    )


x = torch.randn(4, 1, 8, 8, requires_grad=True)
y = torch.tensor([0, 1, 2, 1])
print(f"TinyResNet parameters: {count_parameters(TinyResNet())}")

## Pattern A: auxiliary loss with `log_forward_pass`

Save one intermediate module output, build an auxiliary loss from it, combine that with
the main task loss, and backpropagate through both graphs.

In [ ]:
torch.manual_seed(7)
model = TinyResNet()
model.zero_grad(set_to_none=True)

model_log = tl.log_forward_pass(
    model,
    x,
    train_mode=True,
    layers_to_save=["block1"],
    random_seed=7,
)
aux_activation = model_log["block1"].activation
main_logits = model(x)
loss = F.cross_entropy(main_logits, y) + 0.05 * aux_activation.square().mean()
loss.backward()

assert aux_activation.grad_fn is not None
require_any_grad(model)
print(f"aux loss pattern ok: loss={loss.item():.4f}")
model_log.cleanup()

## Pattern B: probing classifier on a frozen backbone

`train_mode=True` preserves your `requires_grad` setup. A frozen backbone can provide
saved features while only the probe receives parameter gradients.

In [ ]:
torch.manual_seed(7)
backbone = TinyResNet()
freeze(backbone)
probe = nn.Linear(4, 3)
backbone.zero_grad(set_to_none=True)
probe.zero_grad(set_to_none=True)

model_log = tl.log_forward_pass(
    backbone,
    x,
    train_mode=True,
    layers_to_save=["block2"],
    random_seed=7,
)
features = model_log["block2"].activation.mean(dim=(2, 3))
logits = probe(features)
loss = F.cross_entropy(logits, y)
loss.backward()

require_no_grad(backbone)
require_any_grad(probe)
print(f"probe pattern ok: backbone grads={sum(p.grad is not None for p in backbone.parameters())}")
model_log.cleanup()

## Pattern C: multi-tap loss

Save several tap points in one pass and make each one part of the loss. Retaining the
activation grads here is only for the assertion; model training does not require it.

In [ ]:
torch.manual_seed(7)
model = TinyResNet()
model.zero_grad(set_to_none=True)

model_log = tl.log_forward_pass(
    model,
    x,
    train_mode=True,
    layers_to_save=["stem", "block1", "block2"],
    random_seed=7,
)
taps = [model_log[name].activation for name in ["stem", "block1", "block2"]]
for tap in taps:
    tap.retain_grad()
loss = sum(tap.square().mean() for tap in taps)
loss.backward()

assert all(tap.grad is not None for tap in taps)
require_any_grad(model.stem)
require_any_grad(model.block1)
require_any_grad(model.block2)
print(f"multi-tap pattern ok: taps={len(taps)}")
model_log.cleanup()

## Pattern D: activation distillation

Freeze a teacher, keep a student trainable, and match saved activations with MSE. Real
systems often distill logits too; activation MSE keeps this tutorial primitive small.

In [ ]:
torch.manual_seed(7)
teacher = TinyTeacher().eval()
student = TinyStudent().train()
freeze(teacher)
teacher.zero_grad(set_to_none=True)
student.zero_grad(set_to_none=True)

teacher_log = tl.log_forward_pass(
    teacher,
    x,
    train_mode=True,
    layers_to_save=["net.block2"],
    random_seed=7,
)
student_log = tl.log_forward_pass(
    student,
    x,
    train_mode=True,
    layers_to_save=["net.block2"],
    random_seed=7,
)
teacher_activation = teacher_log["net.block2"].activation
student_activation = student_log["net.block2"].activation
loss = F.mse_loss(student_activation, teacher_activation)
loss.backward()

require_no_grad(teacher)
require_any_grad(student)
print(f"distillation pattern ok: mse={loss.item():.4f}")
teacher_log.cleanup()
student_log.cleanup()

## Fastlog and replay paths

`tl.fastlog.Recorder` is useful for rollout-style loops. `ModelLog.save_new_activations`
is useful when you want one exhaustive graph pass and repeated same-graph replays. Both
use `train_mode=True` for graph-connected RAM payloads.

In [ ]:
torch.manual_seed(7)
model = TinyResNet()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
losses: list[float] = []

with tl.fastlog.Recorder(model, train_mode=True) as recorder:
    for step in range(3):
        batch = torch.randn(4, 1, 8, 8, requires_grad=True)
        targets = torch.tensor([(step + offset) % 3 for offset in range(4)])
        output = recorder.log(batch, sample_id=step)
        loss = F.cross_entropy(output, targets)
        loss.backward()
        losses.append(float(loss.item()))
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

relu_payloads = [
    record.ram_payload
    for record in recorder.recording
    if record.ctx.layer_type == "relu" and record.ram_payload is not None
]
assert len(losses) == 3
assert recorder.recording.n_passes == 3
assert all(payload.grad_fn is not None for payload in relu_payloads)
print(f"Recorder passes={recorder.recording.n_passes}, last loss={losses[-1]:.4f}")

In [ ]:
torch.manual_seed(7)
model = TinyResNet()
model_log = tl.log_forward_pass(model, x, layers_to_save="all", random_seed=7)
replay_x = torch.randn(4, 1, 8, 8, requires_grad=True)
model.zero_grad(set_to_none=True)

model_log.save_new_activations(
    model,
    replay_x,
    layers_to_save=["block1"],
    train_mode=True,
    random_seed=7,
)
replay_activation = model_log["block1"].activation
loss = replay_activation.square().mean()
loss.backward()

assert replay_activation.grad_fn is not None
require_any_grad(model)
print("save_new_activations replay kept grad:", replay_activation.grad_fn is not None)
model_log.cleanup()

## Gotchas with executable checks

Replay expects the same graph shape as the exhaustive pass, fastlog respects explicit
capture defaults, and slow/replay training capture is RAM-only. The next cells show the
errors and recovery paths directly.

In [ ]:
class ShapeBranch(nn.Module):
    """Model whose graph changes when the feature dimension changes."""

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run a shape-dependent operation sequence."""

        out = x + 1
        if x.shape[1] == 4:
            out = torch.relu(out)
        return out * 2


torch.manual_seed(7)
shape_model = ShapeBranch()
model_log = tl.log_forward_pass(
    shape_model,
    torch.ones(2, 4, requires_grad=True),
    detach_saved_tensors=True,
    random_seed=7,
)
original_layer_flags = [layer.detach_saved_tensor for layer in model_log]
try:
    model_log.save_new_activations(
        shape_model,
        torch.ones(2, 5, requires_grad=True),
        train_mode=True,
        random_seed=7,
    )
except ValueError as exc:
    print(type(exc).__name__)
    print(str(exc).splitlines()[0])
else:
    raise AssertionError("Expected graph mismatch from shape-dependent replay")

assert model_log.detach_saved_tensors is True
assert model_log.train_mode is False
assert [layer.detach_saved_tensor for layer in model_log] == original_layer_flags
print("dual restore ok")
model_log.cleanup()

In [ ]:
torch.manual_seed(7)
model = TinyResNet()
recording = tl.fastlog.record(model, x, train_mode=True)
print("promoted default payloads:", sum(record.ram_payload is not None for record in recording))

recording = tl.fastlog.record(model, x, train_mode=True, default_op=False)
op_payloads = sum(
    record.ram_payload is not None and record.ctx.kind == "op" for record in recording
)
module_payloads = sum(
    record.ram_payload is not None and record.ctx.kind == "module_exit" for record in recording
)
print("explicit default_op=False:", {"op": op_payloads, "module_exit": module_payloads})

try:
    tl.fastlog.record(
        model,
        x,
        train_mode=True,
        default_op=tl.fastlog.CaptureSpec(keep_grad=False),
    )
except tl.TrainingModeConfigError as exc:
    print(type(exc).__name__)
    print(str(exc).splitlines()[0])
else:
    raise AssertionError("Expected keep_grad=False to conflict with train_mode=True")

In [ ]:
torch.manual_seed(7)
model = TinyResNet()
with tempfile.TemporaryDirectory() as tmpdir:
    tmp_path = Path(tmpdir)
    try:
        tl.log_forward_pass(
            model,
            x,
            train_mode=True,
            layers_to_save="all",
            save_activations_to=tmp_path / "streamed.tl",
        )
    except tl.TrainingModeConfigError as exc:
        print(type(exc).__name__)
        print(str(exc).splitlines()[0])
    else:
        raise AssertionError("Expected train_mode disk streaming to fail")

    model_log = tl.log_forward_pass(model, x, train_mode=True, layers_to_save=["block1"])
    inspect_path = tmp_path / "inspection_only.tl"
    tl.save(model_log, inspect_path)
    loaded = tl.load(inspect_path)
    print("inspection bundle train_mode after load:", loaded.train_mode)
    model_log.cleanup()
    loaded.cleanup()

In [ ]:
torch.manual_seed(7)
backbone = TinyResNet()
freeze(backbone)
probe = nn.Linear(4, 3)
optimizer = torch.optim.SGD(probe.parameters(), lr=0.5)
loop_x = torch.randn(8, 1, 8, 8, requires_grad=True)
loop_y = torch.tensor([0, 1, 2, 0, 1, 2, 0, 1])
losses: list[float] = []

for _step in range(5):
    optimizer.zero_grad(set_to_none=True)
    model_log = tl.log_forward_pass(
        backbone,
        loop_x,
        train_mode=True,
        layers_to_save=["block2"],
        random_seed=7,
    )
    features = model_log["block2"].activation.mean(dim=(2, 3))
    loss = F.cross_entropy(probe(features), loop_y)
    loss.backward()
    optimizer.step()
    losses.append(float(loss.item()))
    model_log.cleanup()

require_no_grad(backbone)
require_any_grad(probe)
assert losses[-1] < losses[0]
print([round(loss, 4) for loss in losses])

## Gotchas to remember

- Disk save plus `train_mode=True` is an error for slow/replay capture; keep trainable
  tensors in RAM, then `tl.save(ml, path)` only for inspection persistence.
- Wrapping the forward in `torch.no_grad()` severs the graph. `torch.inference_mode()` is
  rejected in training mode.
- Training-mode capture pins the autograd graph until `backward()` completes. Do not keep
  a long Python list of training-mode logs or recordings.
- DDP-wrapped models are unwrapped to `.module`; gradients are rank-local and multi-rank
  synchronization is your responsibility.
- Autocast and mixed precision work: saved tensors keep the dtype produced by the actual
  forward.
- Gradient checkpointing works, but recompute operations are not separately visible to
  TorchLens.
- Frozen-backbone correctness is guaranteed by `train_mode=True`; default capture may
  temporarily mutate `requires_grad` during logging.

## Not supported in this training feature

Disk-only training capture, true DDP forward semantics, FSDP, functorch / `vmap`-wrapped
forwards, `inference_mode`-wrapped forwards, and `torch.compile`-wrapped models are not
supported training targets.

## Migration and more references

Existing code that explicitly uses `detach_saved_tensors=False` keeps working:

```python
ml = tl.log_forward_pass(model, x, layers_to_save=["block1"], detach_saved_tensors=False)
loss = ml["block1"].activation.square().mean()
loss.backward()
```

For new training code, prefer the explicit guardrailed form:

```python
ml = tl.log_forward_pass(model, x, layers_to_save=["block1"], train_mode=True)
loss = ml["block1"].activation.square().mean()
loss.backward()
```

`train_mode=True` also preserves frozen parameters and rejects invalid disk/inference
contexts. See the `log_forward_pass`, `ModelLog.save_new_activations`, and
`tl.fastlog.record` docstrings for API details, and use `notebooks/fastlog_tutorial.ipynb`
for capture-only predicate workflows.